# Pràctica 2: Recomanador Heurístic

Nom dels alumnes del grup:


Com fer i estructurar el codi per fer una bona pràctica: 

+ Definir els paràmetres (input) i el retorn (output) de les funcions de forma clara
+ Definir un ordre per defecte dels usuaris/items. Per exemple, si demanen "el film més ben puntuat", ha de ser el que té un 5 i ID més baix.
+ És MOLT important que la funció que calcula la similitud entre les puntuacions en comú de dos usuaris sigui ràpida!
+ Feu unit-tests de Python

## 1. INTRODUCCIÓ

### 1.1. Abans de començar...

**\+ A més a més de les que ja es troben presents en la 1a cel·la i funcions natives de Python, durant la pràctica, només es podran fer servir les següents llibreries**:

`Pandas, Numpy, Itertools`

**\+ No es poden modificar les definicions de les funcions donades, ni canviar els noms de les variables i paràmetres ja donats**

Això no implica però que els hàgiu de fer servir. És a dir, que la funció tingui un paràmetre anomenat `df` no implica que l'hàgiu de fer servir, si no ho trobeu convenient.

**\+ En les funcions, s'especifica què serà i de quin tipus cada un dels paràmetres, cal respectar-ho**

Per exemple, les funcions tindran [pydoc](https://docs.python.org/3/library/pydoc.html) i allà s'especificarà el paràmetre: `df` sempre serà indicatiu del `Pandas.DataFrame` de les dades.

### 1.2. Dades: puntuacions de pel·licules

La base de dades [movielens-1M](http://www.grouplens.org/node/73) conté 1,000,209 puntuacions de 3.900 pel·lícules fetes l'any 2000 per 6.040 usuaris anònims del recomanador online [MovieLens](http://www.movielens.org/). 

El consum total de tots els usuaris s'hi pot trobar al document ``ratings.dat`` el format següent:

    UserID::MovieID::Rating::Timestamp

- **UserID** de l'usuari, amb id's entre 1 i 6040 
- **MovieID** de la pel·licula, amb id's entre 1 i 3952
- **Rating** d'un usuari per una pel·licula, en una escala de 1 (menys) a 5 (més) estrelles.
- **Timestamp** que representa quan aquest usuari va puntuar la pel·licula, representat en segons.

La base de dades original està filtrada de manera que cada usuari té com a mínim 20 puntuacions.

### 1.3. Dades: usuaris



Al fitxer ``users.dat`` hi trobem la informació referent a cadascun dels usuaris en el següent format:

        UserID::Gender::Age::Occupation::Zip-code

- **Gender** ve donat per "M" per home i "F" per dona.
- **Age** està representada de la següent forma:

	*  1:  "Under 18"
	* 18:  "18-24"
	* 25:  "25-34"
	* 35:  "35-44"
	* 45:  "45-49"
	* 50:  "50-55"
	* 56:  "56+"

- **Occupation** es tria entre les següents opcions:

	*  0:  "other" or not specified
	*  1:  "academic/educator"
	*  2:  "artist"
	*  3:  "clerical/admin"
	*  4:  "college/grad student"
	*  5:  "customer service"
	*  6:  "doctor/health care"
	*  7:  "executive/managerial"
	*  8:  "farmer"
	*  9:  "homemaker"
	* 10:  "K-12 student"
	* 11:  "lawyer"
	* 12:  "programmer"
	* 13:  "retired"
	* 14:  "sales/marketing"
	* 15:  "scientist"
	* 16:  "self-employed"
	* 17:  "technician/engineer"
	* 18:  "tradesman/craftsman"
	* 19:  "unemployed"
	* 20:  "writer"

Els usuaris han donat la informació voluntariament. Així doncs, la informació d'alguns usuaris pot estar buida.


### 1.4. Dades: pel·lícules



Al fitxer ``movies.dat`` hi trobem la informació referent a cadascuna de les películes en el següent format:

        MovieID::Title::Genres

- **Titles** són identics als titols de la base de dades IMDB, incloent l'any de llançament.
- **Genres** de les películes, que estan separats pel símbol "|" i estan seleccionats d'entre els següents:

	* Action
	* Adventure
	* Animation
	* Children's
	* Comedy
	* Crime
	* Documentary
	* Drama
	* Fantasy
	* Film-Noir
	* Horror
	* Musical
	* Mystery
	* Romance
	* Sci-Fi
	* Thriller
	* War
	* Western

Algunes películes poden tenir l'ID malament degut a duplicats accidentals.

Les películes s'han entrat manualment, així que poden existir altres inconsistencies. 

## 2. Exploració de les dades

### 2.1 Descarregar i llegir dades

+ Baixa't els fitxers que composen la base de dades i els còpies al teu directori de treball. 

In [15]:
# executeu aquesta cel·la per baixar les dades d'internet
# al campus virtual hi ha un fitxer que podeu baixar també
import os
if os.path.isfile("/etc/password.txt") == False:
    os.system('wget -nc http://files.grouplens.org/datasets/movielens/ml-1m.zip')
    os.system('unzip ml-1m.zip')

+ Llegeix les tres taules de la base de dades en tres DataFrames de pandas amb aquest codi:

In [17]:
import math
import numpy as np
import pandas as pd
import datetime
import itertools
from tqdm.notebook import trange, tqdm
import matplotlib.pyplot as plt

In [18]:
unames = ['user_id', 'gender', 'age', 'occupation', 'zip']
users = pd.read_table('ml-1m/users.dat', sep='::', header=None, names=unames, engine='python')
rnames = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_table('ml-1m/ratings.dat', sep='::', header=None, names=rnames, engine='python')
mnames = ['movie_id', 'title', 'genres']
movies = pd.read_table('ml-1m/movies.dat', sep='::', header=None, names=mnames, engine='python', encoding='latin-1')

### 2.2 Inspecció de les taules

In [20]:
users[:5]

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [21]:
users[-5:]

,user_id,gender,age,occupation,zip
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060
6039,6040,M,25,6,11106


In [22]:
ratings[-5:]

,user_id,movie_id,rating,timestamp
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648
1000208,6040,1097,4,956715569


In [23]:
ratings[:5]

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [24]:
ratings.sort_values('movie_id')[:5]

,user_id,movie_id,rating,timestamp
427702,2599,1,4,973796689
1966,18,1,4,978154768
683688,4089,1,5,965428947
596207,3626,1,4,966594018
465902,2873,1,5,972784317


In [25]:
movies[:5]

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [26]:
ratings[:5]

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


### 2.3 Exemple: Com extreure informació d'un DataFrame.

Suposa que volem calcular les **puntuacions mitjanes d'una pel·licula per sexe o edat**, dades que estan a frames diferents.

El primer pas a obtenir una única estructura que contingui tota la informació. Per fer-ho podem usar la funció ``merge`` de pandas. Aquesta funció infereix automàticament quines columnes ha d'usar per fer el ``merge`` basant-se en els noms que fan intersecció.

Reviseu aquests conceptes de pandas: https://pandas.pydata.org/docs/user_guide/merging.html

In [29]:
data = pd.merge(pd.merge(ratings, users), movies)

# Visualitzem la taula ordenada per identificador d'usuari
data.sort_values(by='user_id')[:10]

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
29,1,745,3,978824268,F,1,10,48067,"Close Shave, A (1995)",Animation|Comedy|Thriller
30,1,2294,4,978824291,F,1,10,48067,Antz (1998),Animation|Children's
31,1,3186,4,978300019,F,1,10,48067,"Girl, Interrupted (1999)",Drama
32,1,1566,4,978824330,F,1,10,48067,Hercules (1997),Adventure|Animation|Children's|Comedy|Musical
33,1,588,4,978824268,F,1,10,48067,Aladdin (1992),Animation|Children's|Comedy|Musical
34,1,1907,4,978824330,F,1,10,48067,Mulan (1998),Animation|Children's
35,1,783,4,978824291,F,1,10,48067,"Hunchback of Notre Dame, The (1996)",Animation|Children's|Musical
36,1,1836,5,978300172,F,1,10,48067,"Last Days of Disco, The (1998)",Drama
37,1,1022,5,978300055,F,1,10,48067,Cinderella (1950),Animation|Children's|Musical


In [30]:
data[data['user_id'] == 2]

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
53,2,1357,5,978298709,M,56,16,70072,Shine (1996),Drama|Romance
54,2,3068,4,978299000,M,56,16,70072,"Verdict, The (1982)",Drama
55,2,1537,4,978299620,M,56,16,70072,Shall We Dance? (Shall We Dansu?) (1996),Comedy
56,2,647,3,978299351,M,56,16,70072,Courage Under Fire (1996),Drama|War
57,2,2194,4,978299297,M,56,16,70072,"Untouchables, The (1987)",Action|Crime|Drama
...,...,...,...,...,...,...,...,...,...,...
177,2,356,5,978299686,M,56,16,70072,Forrest Gump (1994),Comedy|Romance|War
178,2,1245,2,978299200,M,56,16,70072,Miller's Crossing (1990),Drama
179,2,1246,5,978299418,M,56,16,70072,Dead Poets Society (1989),Drama
180,2,3893,1,978299535,M,56,16,70072,Nurse Betty (2000),Comedy|Thriller


La funció ``iloc`` ens permet obtenir un subconjunt de files i/o columnes indexades per un enter:

In [32]:
data.iloc[3:5]

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy


Els índexs Booleans ens permeten seleccionar una part de la taula que compleix una condició.

In [34]:
# comptem quin tant per cent de ratings estan fets per una dona

print(data[data['gender']=='F']['rating'].count()/float(data['rating'].count())*100, '%')

24.638850480249626 %


Per obtenir les **puntuacions mitjanes de cada pel·licula agrupada per edat** podem usar el mètode ``pivot_table`` que és una forma de "canviar" la forma de la taula especificant quin valor agregat (mitjançant una funció predefinida) hi volem en funció dels valors de dues columnes.

Reviseu aquests conceptes: 
+ https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html
+ https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html#pandas.DataFrame.pivot_table

In [36]:
mean_ratings = data.pivot_table(values= 'rating', index='title', columns='age', aggfunc='mean')
mean_ratings[:10]

age,1,18,25,35,45,50,56
title,,,,,,,
"$1,000,000 Duck (1971)",NaN,3.000000,3.090909,3.133333,2.000000,2.750000,NaN
'Night Mother (1986),2.000000,4.666667,3.423077,2.904762,3.833333,3.555556,4.333333
'Til There Was You (1997),3.500000,2.500000,2.666667,2.900000,2.333333,2.500000,2.666667
"'burbs, The (1989)",4.500000,3.244444,2.652174,2.818182,2.545455,3.208333,2.666667
...And Justice for All (1979),3.000000,3.428571,3.724138,3.657143,4.100000,3.551724,3.928571
1-900 (1994),NaN,NaN,2.000000,NaN,NaN,NaN,3.000000
10 Things I Hate About You (1999),3.745455,3.415020,3.432950,3.102941,3.258065,3.629630,4.000000
101 Dalmatians (1961),3.514286,3.295082,3.613757,3.826087,3.976744,3.650000,3.190476
101 Dalmatians (1996),3.088235,2.467742,2.928571,3.279570,3.482759,3.400000,3.555556


Per obtenir les **puntuacions mitjanes de cada pel·licula agrupada per sexe**:

In [38]:
mean_ratings = data.pivot_table('rating', index='title',columns='gender', aggfunc='mean')
mean_ratings[:10]

gender,F,M
title,,
"$1,000,000 Duck (1971)",3.375000,2.761905
'Night Mother (1986),3.388889,3.352941
'Til There Was You (1997),2.675676,2.733333
"'burbs, The (1989)",2.793478,2.962085
...And Justice for All (1979),3.828571,3.689024
1-900 (1994),2.000000,3.000000
10 Things I Hate About You (1999),3.646552,3.311966
101 Dalmatians (1961),3.791444,3.500000
101 Dalmatians (1996),3.240000,2.911215


Si volgéssim fer càlculs només sobre les pel·licules que han rebut **al menys** 250 puntuacions, primer hem de construir una taula amb el nombre d'avaluacions de cada títol. Per fer-ho, agruparem les dades per títol (amb el mètode ``groupby``) i usarem ``size()``.

Reviseu aquest concepte: 

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html

El mètode ``groupby`` implenta un o més d'aquests processos:

+ Dividir les dades segons algun criteri.
+ Aplicar una funció a cada grup.
+ Combinar els resultats en una estructura de dades.

In [40]:
ratings_by_title = data.groupby('title').size()
print(ratings_by_title)

title
$1,000,000 Duck (1971)                         37
'Night Mother (1986)                           70
'Til There Was You (1997)                      52
'burbs, The (1989)                            303
...And Justice for All (1979)                 199
                                             ... 
Zed & Two Noughts, A (1985)                    29
Zero Effect (1998)                            301
Zero Kelvin (Kjærlighetens kjøtere) (1995)      2
Zeus and Roxanne (1997)                        23
eXistenZ (1999)                               410
Length: 3706, dtype: int64


Llavors podem crear un índex amb els títols amb més de 250 avaluacions.

In [42]:
active_titles = ratings_by_title.index[ratings_by_title >= 250]
active_titles

Index([''burbs, The (1989)', '10 Things I Hate About You (1999)',
       '101 Dalmatians (1961)', '101 Dalmatians (1996)', '12 Angry Men (1957)',
       '13th Warrior, The (1999)', '2 Days in the Valley (1996)',
       '20,000 Leagues Under the Sea (1954)', '2001: A Space Odyssey (1968)',
       '2010 (1984)',
       ...
       'X-Men (2000)', 'Year of Living Dangerously (1982)',
       'Yellow Submarine (1968)', 'You've Got Mail (1998)',
       'Young Frankenstein (1974)', 'Young Guns (1988)',
       'Young Guns II (1990)', 'Young Sherlock Holmes (1985)',
       'Zero Effect (1998)', 'eXistenZ (1999)'],
      dtype='object', name='title', length=1216)

L'índex de títols que reben al menys 250 puntuacions es pot fer servir per seleccionar les files de ``mean_ratings``: 

In [44]:
mean_ratings = mean_ratings.loc[active_titles]
mean_ratings

gender,F,M
title,,
"'burbs, The (1989)",2.793478,2.962085
10 Things I Hate About You (1999),3.646552,3.311966
101 Dalmatians (1961),3.791444,3.500000
101 Dalmatians (1996),3.240000,2.911215
12 Angry Men (1957),4.184397,4.328421
...,...,...
Young Guns (1988),3.371795,3.425620
Young Guns II (1990),2.934783,2.904025
Young Sherlock Holmes (1985),3.514706,3.363344


Per veure els films més valorats per les dones, podem ordenar per la columna F de forma descendent:

In [46]:
top_female_ratings = mean_ratings.sort_values(by='F', ascending=False)
top_female_ratings[:10]

gender,F,M
title,,
"Close Shave, A (1995)",4.644444,4.473795
"Wrong Trousers, The (1993)",4.588235,4.478261
Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),4.572650,4.464589
Wallace & Gromit: The Best of Aardman Animation (1996),4.563107,4.385075
Schindler's List (1993),4.562602,4.491415
"Shawshank Redemption, The (1994)",4.539075,4.560625
"Grand Day Out, A (1992)",4.537879,4.293255
To Kill a Mockingbird (1962),4.536667,4.372611
Creature Comforts (1990),4.513889,4.272277


Suposem ara que volem les pel·licules que estan valorades de forma més diferent entre homes i dones. Una forma d'obtenir-ho és afegir una columna a ``mean_ratings`` que contingui la diferència en mitjana i llavors ordenar:

In [48]:
mean_ratings['diff'] = mean_ratings['M'] - mean_ratings['F']

In [49]:
print(np.nan + 9.0) 

nan


Ordenant per ``diff`` ens dóna les pel·licules ben valorades per les dones que presenten més diferència entre homes i dones:

In [51]:
sorted_by_diff = mean_ratings.sort_values(by='diff')
sorted_by_diff[:15]

gender,F,M,diff
title,,,
Dirty Dancing (1987),3.790378,2.959596,-0.830782
Jumpin' Jack Flash (1986),3.254717,2.578358,-0.676359
Grease (1978),3.975265,3.367041,-0.608224
Little Women (1994),3.870588,3.321739,-0.548849
Steel Magnolias (1989),3.901734,3.365957,-0.535777
Anastasia (1997),3.800000,3.281609,-0.518391
"Rocky Horror Picture Show, The (1975)",3.673016,3.160131,-0.512885
"Color Purple, The (1985)",4.158192,3.659341,-0.498851
"Age of Innocence, The (1993)",3.827068,3.339506,-0.487561


Invertint l'ordre de les files i fent un ``slicing`` de les 15 files superiors obtenim les pel·licules ben valorades pels homes que no han agradat a les dones: 

In [53]:
sorted_by_diff[::-1][:15]

gender,F,M,diff
title,,,
"Good, The Bad and The Ugly, The (1966)",3.494949,4.221300,0.726351
"Kentucky Fried Movie, The (1977)",2.878788,3.555147,0.676359
Dumb & Dumber (1994),2.697987,3.336595,0.638608
"Longest Day, The (1962)",3.411765,4.031447,0.619682
"Cable Guy, The (1996)",2.250000,2.863787,0.613787
Evil Dead II (Dead By Dawn) (1987),3.297297,3.909283,0.611985
"Hidden, The (1987)",3.137931,3.745098,0.607167
Rocky III (1982),2.361702,2.943503,0.581801
Caddyshack (1980),3.396135,3.969737,0.573602


Si volguéssim les pel·licules que han generat puntuacions més discordants, independentment del gènere, podem fer servir la variança o la desviació estàndard de les puntuacions: 

In [55]:
rating_std_by_title = data.groupby('title')['rating'].std()

rating_std_by_title = rating_std_by_title.loc[active_titles]
rating_std_by_title.sort_values(ascending=False)[:10]

title
Dumb & Dumber (1994)                     1.321333
Blair Witch Project, The (1999)          1.316368
Natural Born Killers (1994)              1.307198
Tank Girl (1995)                         1.277695
Rocky Horror Picture Show, The (1975)    1.260177
Eyes Wide Shut (1999)                    1.259624
Evita (1996)                             1.253631
Billy Madison (1995)                     1.249970
Fear and Loathing in Las Vegas (1998)    1.246408
Bicentennial Man (1999)                  1.245533
Name: rating, dtype: float64

### Important: Temes de rendiment

Fixeu-vos en el comportament de Python en aquests tres exmepls (que tenen el mateix output). Identifiqueu l'origen de les diferències i actueu en conseqüència:

In [57]:
# Aquesta cel·la pot trigar uns segons a executar-se

%timeit data['title'] 
print(type(data['title']))
%timeit data.title 
print(type(data.title))
%timeit data[['title']] 
print(type(data[['title']]))

4.2 μs ± 276 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
<class 'pandas.core.series.Series'>
7.93 μs ± 358 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
<class 'pandas.core.series.Series'>
13.7 ms ± 645 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
<class 'pandas.core.frame.DataFrame'>


## 3. EXERCICIS

### 3.1. EXERCICI A

+ Donada la taula ``data`` tal i com es defineix a continuació, calcula la puntuació mitjana de cada usuari i guarda-la a un ``df`` anomenat ``users_mean_rating``. 

In [60]:
data_folder = 'ml-1m'

In [61]:
unames = ['user_id', 'gender', 'age', 'occupation', 'zip']
users = pd.read_table(f'{data_folder}/users.dat', sep='::', header=None, names=unames, engine='python')
rnames = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_table(f'{data_folder}/ratings.dat', sep='::', header=None, names=rnames, engine='python')
mnames = ['movie_id', 'title', 'genres']
movies = pd.read_table(f'{data_folder}/movies.dat', sep='::', header=None, names=mnames, engine='python',encoding='latin-1')

data = pd.merge(pd.merge(ratings, users), movies)

# Basicament agrupem les dades per user i calculem la mitjana de la columna rating
users_mean_rating = data.groupby('user_id')['rating'].mean()

In [62]:
users_mean_rating

user_id
1       4.188679
2       3.713178
3       3.901961
4       4.190476
5       3.146465
          ...   
6036    3.302928
6037    3.717822
6038    3.800000
6039    3.878049
6040    3.577713
Name: rating, Length: 6040, dtype: float64

In [63]:
data

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy
...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy
1000205,6040,1094,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War
1000206,6040,562,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama


+ Quina és la pel·lícula més ben puntuada (en mitja) pels usuaris? (Guarda aquest valor en una variable de tipus ``string`` anomenada ``best_movie_rating`` ). 

In [65]:
# Obtenim el rating de cada pel·lícula
movie_ranking = data.groupby('movie_id')['rating'].mean()

# Ens quedem amb l'índex de l'element amb valor máxim, també guardem la seva puntuació
best_movie_rating = movie_ranking.idxmax()
best_ranking = movie_ranking.loc[best_movie_rating]

print(movie_ranking)
print(best_movie_rating)
print(best_ranking)

movie_id
1       4.146846
2       3.201141
3       3.016736
4       2.729412
5       3.006757
          ...   
3948    3.635731
3949    4.115132
3950    3.666667
3951    3.900000
3952    3.780928
Name: rating, Length: 3706, dtype: float64
787
5.0


+ Mira si hi ha més pel·licules amb la mateixa puntuació de la més ben puntuada.

In [67]:
# Obtenim els elements amb el ranking de l'element top
top_rated_movies = movie_ranking[movie_ranking == best_ranking]
print(top_rated_movies)

movie_id
787     5.0
989     5.0
1830    5.0
3172    5.0
3233    5.0
3280    5.0
3382    5.0
3607    5.0
3656    5.0
3881    5.0
Name: rating, dtype: float64


+ Busca ara aquella pel·lícula, d'entre les que tenen 5 com a puntuació mitjana, que hagi rebut més valoracions i guarda-la a una variable anomenada ``best_movie_rating_maxviews``. Aixi tindrem la pel·licula més ben puntuada per més usuaris. 

In [69]:
# Obtenim les pelis segons si estan en el ranking anterior
# Aleshores agrupem per movie_id i comptem quantes files hi ha
top_rated_movie_counts = data[data['movie_id'].isin(top_rated_movies.index)].groupby('movie_id').size()

# Obtenim líndex de la peli amb millor puntuació
best_movie_rating_maxviews = top_rated_movie_counts.idxmax()

print(best_movie_rating_maxviews)

787


### 3.2. EXERCICI B

+ Defineix una funció anomenada ``top_movie`` que donat un usuari ens retorni quina és la pel·lícula millor puntuada.


In [71]:
def top_movie(dataFrame, usr):
    # Ens quedem amb les files del user que ens interessa
    rated_films_by_user = dataFrame[dataFrame['user_id'] == usr]
    
    # Guardem els ratings de l'usuari, serviex per estalviar càlculs
    rating_by_user = rated_films_by_user['rating']
    
    # Obtenim la puntuació més alta d'aquest user
    top_rating_by_user = rating_by_user.max()

    # Obtenim totes les pelis amb la puntuació més alta
    top_movies = rated_films_by_user[rating_by_user == top_rating_by_user]

    # En cas de només voler una peli podem fer 
    # top_film_by_user = rating_by_user.idxmax()
    # top_movies = rated_films_by_user.loc[top_film_by_user]
    
    return top_movies[['title', 'movie_id']]

    # top_movie_row = rated_films_by_user.loc[] 

print(top_movie(data, 1))

                                     title  movie_id
0   One Flew Over the Cuckoo's Nest (1975)      1193
4                     Bug's Life, A (1998)      2355
6                           Ben-Hur (1959)      1287
7                Christmas Story, A (1983)      2804
10             Beauty and the Beast (1991)       595
14              Sound of Music, The (1965)      1035
18                       Awakenings (1990)      3105
22               Back to the Future (1985)      1270
23                 Schindler's List (1993)       527
25                       Pocahontas (1995)        48
36          Last Days of Disco, The (1998)      1836
37                       Cinderella (1950)      1022
39                        Apollo 13 (1995)       150
40                        Toy Story (1995)         1
41                         Rain Man (1988)      1961
45                     Mary Poppins (1964)      1028
46                            Dumbo (1941)      1029
48              Saving Private Ryan (1998)    

### 3.3. EXERCICI C

+ Construeix una funció que donat el dataframe ``data`` et retorni un altre dataframe ``df_counts``amb el valor que cada usuari li ha donat a una peli. Això ho farem creant un dataframe on les columnes estiguin indexades per `movie_id`, les files per `user_id` i els valors siguin el rating donat.

In [74]:
def build_counts_table(df):
    """
    Retorna un dataframe on les columnes són els `movie_id`, les files `user_id` i els valors
    la valoració que un usuari ha donat a una peli d'un `movie_id`
    
    :param df: DataFrame original 
    :return: DataFrame descrit adalt
    """
    
    # Apliquem la funció pivot a user_id i movie_id per tenir les puntuacions
    return df.pivot_table(values='rating', index='user_id', columns='movie_id')


In [75]:
df_counts = build_counts_table(data)
df_counts

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6036,NaN,NaN,NaN,2.0,NaN,3.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6037,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6038,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


+ Fés una funció `get_count` que donada la taula anterior i dos id's (usuari i peli), extregui el valor donat:

In [77]:
def get_count(df, user_id, movie_id):
    """
    Retorna la valoració que l'usuari 'user_id' ha donat de 'movie_id'
    
    :param df: DataFrame retornat per `build_counts_table`
    :param user_id: ID de l'usuari
    :param movie_id: ID de la peli
    :return: Enter amb la valoració de la peli
    """

    # Utilitzem iloc per obtenir el valor que volem
    return df.iloc[user_id, movie_id]

get_count(df_counts, 1, 1)

nan

### 3.4. EXERCICI D

In [79]:
data.nunique()

user_id         6040
movie_id        3706
rating             5
timestamp     458455
gender             2
age                7
occupation        21
zip             3439
title           3706
genres           301
dtype: int64

In [80]:
unique_movies = pd.unique(data['movie_id'])
unique_movies.max()

3952

Si observem el nombre total d'usuaris únics i de pel.licules úniques, podem veure que els id's dels usuaris van de 1 a 6040. Normalment volem índexos que comencin al nombre 0, anant de 0 a 6039. 

+ Explora els índexos de les pel·licules. **Quin problema hi ha amb els indexos de les pel·licules??**

> **Resposta**
>
> La primera comanda serveix per obtenir la quantitat d’elements únics de cada columna del ``DataFrame``. La segona comanda obté el valor màxim en la columna de ``movie_id``. Així, la primera comanda ens diu que hi ha un total de $3706$ pel·lícules diferents, però amb la segona veiem que el ``movie_id`` més alt és $3952$. Això indica que **hi ha índexs que no existeixen**: si tenim $3706$ pel·lícules úniques i el ``movie_id`` més elevat és $3952$, pel Teorema del Colomar (https://es.wikipedia.org/wiki/Principio_del_palomar), hi ha valors entre $1$ i $3952$ que no tenen cap pel·lícula assignada. Això significa que els índexs no són continus.


+ Usant la funció `pd.Categorical(*).codes`, re-indexa els id's dels usuaris i de les pelis perquè vagin de 0 a 6039 i de 0 a 3705 respectivament:

In [84]:
# Utilitzem pd.Categorical() per reindexar
data['user_id'] = pd.Categorical(data['user_id']).codes
data['movie_id'] = pd.Categorical(data['movie_id']).codes

In [85]:
# data[data['user_id'] == 2]

+ Per comprovar que tot sigui correcte i guardar correctament la taula **df_counts**, torna a calcular i visualitza ``df_counts``:

In [87]:
df_counts = build_counts_table(data)
df_counts

movie_id,0,1,2,3,4,5,6,7,8,9,...,3696,3697,3698,3699,3700,3701,3702,3703,3704,3705
user_id,,,,,,,,,,,,,,,,,,,,,
0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6035,NaN,NaN,NaN,2.0,NaN,3.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6036,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6037,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3.5. EXERCICI E



+ Escriu una funció `distEuclid(x,y)`  que implementi la **distància** Euclidiana entre dos vectors usant funcions de numpy. 

+ Escriu la funció `simEuclid(U1, U2)` que calculi la **similitud** entre dos vectors segons la fòrmula següent (on $n$ és un factor de normalització). Ho fem així perquè si dos usuaris tenen moltes pel·licules en comú, volem que la similitud entre aquests usuaris sigui major que el de dos usuaris que només n'han vist una en comú. Si els vectors estan buits, retornar 0.

    $$d =  \frac{1}{(1+distEuclid(U1, U2))} \times \frac{len(U1)}{n} $$

+ Escriu la funció `simUsuaris(df, U1, U2)` per retorna la sembalça de dos usuaris a partir del `df_counts`, tenint en compte les puntuacions que tenen en comú, fent servir les dues funcions anteriors.
    
+ Avalueu amb la funció ``%timeit`` quant triguen aquests càlculs per un parell d'usuaris.   

> *Nota: Alguns d'aquests exercicis tenen temps de càlcul de l'ordre de minuts sobre tota la base de dades. Per desenvolupar els algorismes és recomanable treballar amb una versió reduïda de la base de dades.* 

Per implementar aquestes funcions únicament es permet l'ús de les funcions:

* `np.sum`
* `np.sqrt`
* `np.power`
* `np.dot`
* `np.linalg.norm`
* `np.mean`

I s'ha de fer **sense bucles**!

In [91]:
num_movies = data.nunique()['movie_id']

def distEuclid(x, y):
    """
    Retorna la distancia euclidiana de dos vectors n-dimensionals.
    
    :param x: Primer vector
    :param y: Segon vector
    :return : Escalar (float) corresponent a la distancia euclidiana
    """
    # Apliquem la comanda de numpy per trobar la distància euclidiana
    return np.linalg.norm(x - y)


def simEuclid(Vec1, Vec2, norm):
    """
    Retorna la sembalça de dos vectors.
    
    :param Vec1: Primer vector
    :param Vec2: Segon vector
    :return : Escalar (float) corresponent a la semblança
    """
    # Com diu l'enunciat, algún vector està buit, retornem 0
    if(len(Vec1) == 0 and len(Vec2) == 0):
        return 0

    # Ens assegurem que la norma no sigui zero
    norm = 1 if norm == 0 else norm

    # Fem el càlcul amb la fòrmula
    return (1 / (1 + distEuclid(Vec1, Vec2))) * (len(Vec1) / norm)

    
def simUsuaris(DataFrame, User1, User2):
    """
    Retorna un score que representa la similitud entre user1 i user2 basada en la distancia euclidiana
    
    :param DataFrame: dataframe que conté totes les dades
    :param User1: id user1
    :param User2: id user2
    :return : Escalar (float) corresponent al score
    """
    # Obtenim les puntuacions de l'usuari
    user1 = DataFrame.iloc[User1]
    user2 = DataFrame.iloc[User2]

    # Traiem NaNs
    ratings_user1 = user1.dropna()
    ratings_user2 = user2.dropna()
    
    # Trobem les reviews en comú
    intersection_indexes = ratings_user1.index.intersection(ratings_user2.index)

    # Vectors de ratings en comú
    v1 = ratings_user1[intersection_indexes].values
    v2 = ratings_user2[intersection_indexes].values

    # Calculem la similitud
    # Considerem que la norma és el total de pel·lícules de la base de dades
    return simEuclid(v1, v2, len(user1))

In [92]:
print(simUsuaris(df_counts, 2, 314))

0.0005396654074473826


In [93]:
# Aqui ha de sortir un valor de l'ordre de microsegons, no milisegons
%timeit simUsuaris(df_counts, 1, 5)

1.33 ms ± 204 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### 3.6. EXERCICI F

En aquest exercici desenvoluparem un sistema de recomanació col·laboratiu **basat en usuaris**. 

La funció principal, ``getRecommendationsUser``, ha de tenir com a entrada una taula de puntuacions, un ``user_id``, el tipus de mesura de similitud (Euclidiana) que volem usar, el nombre `m` d'usuaris semblants que volem per fer la recomanació i el nombre ``n`` de recomanacions que volem. 

Exemple: ``getRecommendationsUser(data, 2, 50, 10, simEuclid)``

Com a sortida ha de donar la llista de les $n$ millors pel·lícules que li podriem recomanar segons la seva semblança amb altres usuaris.

> *Nota 1: S'ha d'evitar comparar ``user_id`` a ell mateix.*

> *Nota 2: Recordeu que en Python podem passar funcions com a paràmetres d'una funció.*

In [95]:
import heapq

#### EXERCICI F.1

+ Computa la *score* de similitud del usuari desitjat (``userID``) respecte tots els altres i retorna un diccionari dels $m$ usuaris més propers i el seu *score* de semblança. Fes servir la matriu `df_counts` Normalitzeu els *scores* de sortida de manera que sumin 1.

In [97]:
def normalize(z):
    z_exp = [math.exp(i) for i in z]
    sum_z_exp = sum(z_exp)
    softmax = [round(i / sum_z_exp, 4) for i in z_exp]
    return softmax

In [98]:
def find_similar_users(DataFrame, userID, m, simfunction):
    """
    Retorna un diccionari de usuaris similars amb les scores corresponents.
    
    :param DataFrame: dataframe que conté totes les dades
    :param userID: usuari respecte al qual fem la recomanació
    :param m: nombre d'usuaris que volem per fer la recomanació
    :param similarity: mesura de similitud
    :return : dictionary
    """
    # Guardem les dimensions del data frame
    h, w = DataFrame.shape
    scores = []
    
    # Com que ja hem filtrar els id del users abans (amb el pd.Categorical) podem iterar directament
    for user in range(h):
        # Ignorem el cas del propi usuari
        if user == userID:
            continue

        # Calculem la similitud entre els dos usuaris
        similarity = simfunction(DataFrame, userID, user)

        # Si a la nostra llista de scores encara queda espai, afegim directament
        if len(scores) < m:
            heapq.heappush(scores, (similarity, user))
        # Sino afegim l'element i eliminem el més petit
        else:
            heapq.heappushpop(scores, (similarity, user))

    # Normalitzem els valors de les similituds
    similarities, users = zip(*scores)
    similarities = normalize(similarities)
    scores = list(zip(similarities, users))
    
    return scores

In [99]:
t = datetime.datetime.now()
sim_dict = find_similar_users(df_counts, 2, 10, simUsuaris)
t = datetime.datetime.now()-t
print(str(t))

0:00:08.828740


In [100]:
sim_dict

[(0.1, 650),
 (0.1, 3625),
 (0.1, 3271),
 (0.1, 1879),
 (0.1, 1903),
 (0.1, 2270),
 (0.1, 1646),
 (0.1, 4276),
 (0.1, 4447),
 (0.1, 5830)]

+ Quan trigaria (en minuts) si ho fem per tots els usuaris?

**Resposta**: l'únic que canviaria és que faríem un `push` per a tot element del heap i el `softmax` hauria de calcular la norma de tots els usuaris. Al no haver d'executar `heappushpop` en cap moment ens estalviem temps, ja que òbviament és més costosa que `heappush`.

+ Anem ara a construir una `matriu` de mida $U \times U$ on cada posició $(i,j)$ indiqui la distància entre l'element $i$ i el $j$. Així doncs, si estàs fent un recomanador basat en usuaris, `matriu[2, 3]` contindrà la similitud entre l'usuari 2 i el 3.

Compareu aquestes dues opcions des del punt de vista de temps de càlcul:

* Feu una funció,  que construeixi la ``similarity_matrix1`` a partir de la distancia entre usuaris com la distància entre els vectors formats pels elements en comú dels dos usuaris.
* Feu una funció que construeixi la ``similarity_matrix2`` d'una forma *aproximada*:
    + Substituïnt els ``nand`` que corresponen a ítems no avaluats per ``0``. 
    + Treballant específicament amb operacions matricials. En aquest link podeu trobar indicacions de com fer-ho: https://jaykmody.com/blog/distance-matrices-with-numpy/

In [105]:
def compute_similitude(fixed_arr, var_arr):
    """
    Donats dos vectors, calcula la similitud entre els subvectors formats 
    pels elements en comú (sense fer servir cap iteració!). 
    Normalitzeu la sortida multiplicant pel nombre de pel·lícules vistes en comú i
    dividint pel nombre total de pelis del dataset
    """
    # Conjunt d'elements comuns (on cap dels dos és NaN)
    common_mask = (~np.isnan(fixed_arr)) & (~np.isnan(var_arr))
    
    # Obtenim els subvectors formats pels elements comuns
    fixed_common = fixed_arr[common_mask]
    var_common = var_arr[common_mask]

    # Calculem la similitud
    return simEuclid(fixed_common, var_common, fixed_arr.size)

In [106]:
# test
vec1 = np.array([1, np.nan, 2, 3, 4])
vec2 = np.array([np.nan, 4, 5, 2, 2])
print(compute_similitude(vec1, vec2))

vec1 = np.array([1, np.nan, 1, 1, 1])
vec2 = np.array([np.nan, 1, 1, 1, 1])
print(compute_similitude(vec1, vec2))

vec1 = np.array([1, 1, 1, 1, 1])
vec2 = np.array([1, 1, 1, 1, 1])
print(compute_similitude(vec1, vec2)*num_movies/5)

0.12653803323572038
0.6
741.2


In [107]:
def similarity_matrix_1(compute_distance, df_counts):
    """
    Retorna una matriu de mida M x M on cada posició 
    indica la similitud entre usuaris (resp. ítems).
    
    :param df_counts: df amb els valor que cada usuari li ha donat a una peli.
    :return : Matriu numpy de mida M x M amb les similituds.
    """
    # Guardem les dimensions del data frame
    h, _ = df_counts.shape

    # Creem una matriu de dimensió h x h
    matrix = np.zeros((h, h))
    
    for user1 in range(h):
        # Tenint en compte que la relació és simétrica, podem estalviar-nos càlculs
        for user2 in range(user1, h):
            # La similitud entre un mateix usuari val 1
            if(user1 == user2):
                matrix[user1, user2] = matrix[user2, user1] = 1
                continue
                
            ratings_user1 = df_counts.iloc[user1]
            ratings_user2 = df_counts.iloc[user2]

            # Omplim la matriu amb la distancia entre ambdós usuaris
            similarity = compute_distance(ratings_user1, ratings_user2)
            matrix[user1, user2] = matrix[user2, user1] = similarity

    return matrix

In [108]:
# Definim un test per fer-ho més còmode
test_df = df_counts.sample(n = 200)

In [109]:
t = datetime.datetime.now()
sim = similarity_matrix_1(compute_similitude, test_df)
t = datetime.datetime.now()-t
print("Temps amb doble for:",str(t))

Temps amb doble for: 0:00:18.527054


In [110]:
def similarity_matrix_2(DataFrame):
    """
    Retorna una matriu de mida M x M on cada posició 
    indica la similitud entre usuaris (resp. ítems).
    Substitueix els nand per 0.

    :return : Matriu numpy de mida M x M amb les similituds.
    """
    # Substituim els NaN per 0 per tal de fer les operacions
    data = DataFrame.fillna(0).values
    
    # Normalitzem els vectors
    norms = np.linalg.norm(data, axis=1, keepdims=True)

    # Substituïm les normes zero per 1 per evitar divisió per zero
    normalized_data = np.divide(data, norms, where=(norms != 0))
    
    # Calculem la matriu de similituds (producte escalar entre vectors)
    similarity_matrix = np.matmul(normalized_data, normalized_data.T)
    
    return similarity_matrix

In [111]:
t = datetime.datetime.now()
sim = similarity_matrix_2(df_counts)
t = datetime.datetime.now()-t
print("Temps matricialment:",str(t))

Temps matricialment: 0:00:02.344045


In [112]:
sim

array([[1.        , 0.09638153, 0.12060981, ..., 0.        , 0.17460369,
        0.13359025],
       [0.09638153, 1.        , 0.1514786 , ..., 0.06611767, 0.0664575 ,
        0.21827563],
       [0.12060981, 0.1514786 , 1.        , ..., 0.12023352, 0.09467506,
        0.13314404],
       ...,
       [0.        , 0.06611767, 0.12023352, ..., 1.        , 0.16171426,
        0.09930008],
       [0.17460369, 0.0664575 , 0.09467506, ..., 0.16171426, 1.        ,
        0.22833237],
       [0.13359025, 0.21827563, 0.13314404, ..., 0.09930008, 0.22833237,
        1.        ]])

+ Ara torna a refer la funció ``find_similar_users`` usant la matriu de distàncies i mira quant triga. Recorda que les scores han d'estar normalitzades!

In [114]:
def find_similar_users(DataFrame, sim_mx, userID, m):
    similarity_vector = sim_mx[userID]
    scores = []
    
    for user, similarity in enumerate(similarity_vector):
        if(user == userID):
            continue
            
        # Si a la nostra llista de scores encara queda espai, afegim directament
        if len(scores) < m:
            heapq.heappush(scores, (similarity, user))
        # Sino afegim l'element i eliminem el més petit
        else:
            heapq.heappushpop(scores, (similarity, user))
    
    # Normalitzem els valors de les similituds
    similarities, users = zip(*scores)
    similarities = normalize(similarities)
    scores = list(zip(similarities, users))

    # Retornem les tuples ordenades pels users
    return sorted(scores, key=lambda x: x[1])

In [115]:
t = datetime.datetime.now()
sim_dict = find_similar_users(df_counts, sim, 2, 10)
t = datetime.datetime.now()-t
print(str(t))

0:00:00.003886


In [116]:
sim_dict

[(0.0988, 310),
 (0.1017, 478),
 (0.0994, 1903),
 (0.0993, 2261),
 (0.099, 2434),
 (0.102, 2999),
 (0.1003, 3499),
 (0.0993, 4319),
 (0.0987, 4888),
 (0.1015, 5690)]

> En l'anterior `find_similar_users` hem passat una norma fixada per paràmetere per a `simUsuaris` que corresponia a la longitud del primer usuari. En aquest segon cas hem calculat de forma dinàmica la norma $n$ utilitzant la mida del vector rebut sense els `NaNs`.

#### EXERCICI F.2

+ Computa les recomanacions per un usuari concret a partir dels scores dels seus $m$ usuaris més propers. 
    + Fes primer una funció ``weighted_average`` que retorni un diccionari del tipus ``{peli_id: score predit}`` amb la puntuació predita de cada ítem a partir de les puntuacions dels $m$ usuaris més propers i de la seva semblança a l'usuari considerat.
    + Fes després una funció ``getRecommendationsUser`` que usant la funció anterior retorni un ``df`` amb els $n$ ítems amb més score i els seus scores.

In [119]:
def weighted_average(DataFrame, user, sim_mx, m):
    """    
    :param DataFrame: dataframe que conté totes les dades
    :param user: usuari al qual fem la recomanació
    :param sim_mx: similarity_matrix
    :param m: nombre d'usuaris semblants a tenir en compte per les recomanacions
    :return: diccionari {peli_id: score predit}
    """
    # Busquem els m usuaris semblants
    similar_users = find_similar_users(DataFrame, sim_mx, user, m)

    # Separem les similarities dels users
    similarities, users = zip(*similar_users)

    # Deixem similarities com una matriu vertical
    similarities = np.array(similarities).reshape(-1, 1)
    
    # Ens quedem amb les puntuacions d'aquests usuaris i
    # eliminem les columnes on tot son NaNs
    selected_users = DataFrame.loc[list(users)].dropna(axis=1, how="all")

    # Calculem la suma de les similarities
    # Utilitzem notna() per tenir True/False i obtenir el recompte
    sim_sum = (selected_users.notna() * similarities).sum(axis=0)
    
    # Multipliquem les similarities pels users del df i fem la suma per pelis
    total = (selected_users * similarities).sum(axis=0)

    # Normalitzem els valors
    total_sim_sum = total / sim_sum

    # Retornem un diccionari del tipus {peli_id: score}
    return total_sim_sum.to_dict()

In [120]:
weighted_average(df_counts, 2, sim, 10)

{0: 4.601887171250753,
 1: 3.5052264808362366,
 5: 4.0,
 8: 1.0,
 9: 4.489105935386927,
 10: 4.673070553163299,
 18: 5.0,
 20: 5.0,
 23: 3.0,
 24: 1.0,
 31: 4.255132699048573,
 33: 4.500736377025038,
 35: 4.498233215547703,
 38: 3.0000000000000004,
 46: 2.505718547986077,
 47: 3.0,
 49: 4.0,
 60: 4.0,
 67: 5.0,
 68: 4.0,
 71: 4.0,
 84: 4.0,
 92: 2.9989902389767753,
 101: 4.498233215547703,
 102: 3.0,
 106: 4.799677224127496,
 108: 3.0000000000000004,
 138: 5.0,
 139: 3.0,
 144: 5.0,
 147: 3.3457943925233655,
 154: 2.66844563042028,
 155: 4.339226150767177,
 159: 3.332211376640861,
 163: 2.9999999999999996,
 179: 2.0,
 190: 2.0,
 202: 4.0,
 216: 4.0,
 228: 4.0,
 232: 3.0,
 246: 3.4982332155477036,
 253: 4.7963000000000005,
 259: 3.0,
 273: 4.0,
 279: 3.0000000000000004,
 283: 3.332211376640861,
 287: 3.2461113898645255,
 294: 2.5040201005025122,
 307: 4.0,
 309: 5.0,
 319: 3.5067064083457526,
 323: 5.0,
 327: 5.0,
 329: 5.0,
 334: 5.0,
 339: 4.397257551669316,
 343: 3.4992412746585737,


In [121]:
def getRecommendationsUser(DataFrame, user, sim_mx, n, m):
    """    
    :param DataFrame: dataframe que conté totes les dades
    :param user: usuari al qual fem la recomanació
    :param sim_mx: similarity_function
    :param n: nombre de pelis a recomanar
    :param m: nombre d'usuaris semblants a tenir en compte per les recomanacions
    :return : dataframe de pel·licules amb els scores.
    """
    # Obtenim les puntuacions predites per cada peli
    predicted_scores = weighted_average(DataFrame, user, sim_mx, m)

    # Obtenim les pelis que ja ha puntuat l'usuari
    user_rated_movies = DataFrame.loc[user].dropna().index.tolist()

    # Eliminem del diccionari les pelis que l'usuari ja ha puntuat
    filtered_scores = {movie: score for movie, score in predicted_scores.items() if movie not in user_rated_movies}

    # Agafem les n millors pelis
    top_n_movies = sorted(filtered_scores.items(), key=lambda item: item[1], reverse=True)[:n]
    
    # Convertim el resultat a un DataFrame
    recommendations = pd.DataFrame(top_n_movies, columns=["movie_id", "predicted_score"])
    
    return recommendations

In [122]:
t = datetime.datetime.now()
user_prediction = getRecommendationsUser(df_counts, 3, sim, 10, 50)
t = datetime.datetime.now()-t
print(str(t))

0:00:00.020865


In [123]:
user_prediction

,movie_id,predicted_score
0,46,5.0
1,49,5.0
2,107,5.0
3,281,5.0
4,309,5.0
5,346,5.0
6,704,5.0
7,731,5.0
8,742,5.0
9,754,5.0


### 3.7. EXERCICI G


A continuació usarem la metrica **Mean Absolute Error (MAE)** per evaluar el nostre sistema. Aquesta mètrica ens permetrà mesurar la diferencia entre dues llistes donat un usuari: 
+ La llista amb els scores reals d'un usuari
+ La llista amb els scores predits per aquest usuari

#### EXERCICI G.1

Anem a crear un conjunt de training i un de test de forma "ingènua":
+ Selecciona de forma aleatòria el 10% dels usuaris i guarda'ls en una llista anomenada ``test_set``.  
+ Guarda la resta en una llista anomenada ``train_set``.
+ Mira quants elements tenen aquestes llistes.

In [127]:
# Utilitzem sample per obtenir el 10% del usuaris
# Utilitzem random_state per tenir uniformitat en els test
test_set = data.sample(frac=0.1, random_state=33)

# Guardem la resta de dades pel training
train_set = data.drop(test_set.index)

In [128]:
print(f"Test set size: {len(test_set)}")
print(f"Train set size: {len(train_set)}")

Test set size: 100021
Train set size: 900188


In [129]:
test_set

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
400926,2388,287,4,974297704,M,25,16,37922,Pulp Fiction (1994),Crime|Drama
959633,5787,737,2,958107471,M,25,0,92646,Independence Day (ID4) (1996),Action|Sci-Fi|War
204964,1260,2956,2,974820994,M,18,4,40205,Any Given Sunday (1999),Drama
208257,1272,228,4,974816465,M,35,2,19123,Ed Wood (1994),Comedy|Drama
844927,5076,2984,3,962653068,M,25,2,20037,"Boys from Brazil, The (1978)",Thriller
...,...,...,...,...,...,...,...,...,...,...
714165,4276,3476,5,989177141,M,35,16,98133,American Pimp (1999),Documentary
980583,5915,1744,1,957459704,M,50,20,48230,Plan 9 from Outer Space (1958),Horror|Sci-Fi
100482,669,2586,4,975627921,M,25,12,30303,Airplane! (1980),Comedy
789015,4724,2257,3,963374723,M,35,5,96707-1321,Howard the Duck (1986),Adventure|Children's|Sci-Fi


In [130]:
train_set

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
0,0,1104,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,0,639,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,0,853,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,0,3177,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,0,2162,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy
...,...,...,...,...,...,...,...,...,...,...
1000204,6039,1019,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy
1000205,6039,1022,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War
1000206,6039,548,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama
1000207,6039,1024,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama


> **Resposta**
>
> Veiem que el `test_set` té $100021$ elements, això és exactament una dècima part del nostre total, $1000209$. Amb el mateix argument, `train_set` té el $90\%$ de les dades, és a dir, $900188$.

+ Què passarà si calculo la matriu de similitud amb ``train_set`` i després intento predir pels usuaris de ``test_set``??

> **Resposta**
>
> Observem que tots dos conjunts (`train_set` i `test_set`) contenen totes les pel·lícules del dataset, però es divideixen segons els usuaris. Això implica que, si calculo la matriu de similitud utilitzant només el `train_set`, estaré obtenint les similituds exclusivament entre els usuaris presents en aquest conjunt. Quan intenti fer prediccions per als usuaris del `test_set`, no disposaré de cap informació rellevant sobre aquests usuaris a la matriu de similitud, ja que no han participat en el càlcul de les similituds. Això dificultarà, o fins i tot farà impossible, realitzar prediccions acurades per als usuaris del test.
> 

#### EXERCICI G.2

Cambien ara la manera de generar els conjunts per no tenir el problema anterior.

+ Seleccionarem aproximadament el 80% de les interaccions de cada usuari de ``test_set`` i les afegirem al ``train_set``. 
+ Podriem ara podem evaluar el sistema?

> Us donem el codi per un usuari donat i vosaltres només heu de crear la funció que, per cada usuari, afageixi el 80% de les intraccions al ``train_set``.

In [136]:
test_set.head()

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
400926,2388,287,4,974297704,M,25,16,37922,Pulp Fiction (1994),Crime|Drama
959633,5787,737,2,958107471,M,25,0,92646,Independence Day (ID4) (1996),Action|Sci-Fi|War
204964,1260,2956,2,974820994,M,18,4,40205,Any Given Sunday (1999),Drama
208257,1272,228,4,974816465,M,35,2,19123,Ed Wood (1994),Comedy|Drama
844927,5076,2984,3,962653068,M,25,2,20037,"Boys from Brazil, The (1978)",Thriller


In [137]:
# Agafem el 20% de les pelis que ha consumit cada usuari de test 
groupby_count = test_set.groupby('user_id')['movie_id'].count()*0.2
groupby_count

user_id
0        0.8
1        1.6
2        1.2
3        0.6
4        5.0
        ... 
6035    15.8
6036     4.2
6037     0.2
6038     2.2
6039     9.6
Name: movie_id, Length: 5969, dtype: float64

Seleccionem la posició 1 i aquest ``user_id`` serà el que usarem pel codi d'exemple (que després haureu de replicar).

In [139]:
groupby_count.reset_index().iloc[1]

user_id     1.0
movie_id    1.6
Name: 1, dtype: float64

In [140]:
n_test_samples = int(groupby_count.reset_index().iloc[1]['movie_id'])
u = groupby_count.reset_index().iloc[1]['user_id']

In [141]:
test_set_user = test_set[test_set['user_id'] == u]
frame_test = test_set_user.sample(n_test_samples)
print("TOTAL SAMPLES OF THE USER: " + str(len(test_set_user)))
print("TOTAL SAMPLES OF THE USER IN TEST SET: " + str(len(frame_test)))

TOTAL SAMPLES OF THE USER: 8
TOTAL SAMPLES OF THE USER IN TEST SET: 1


In [142]:
len(test_set_user.index)

8

In [143]:
frame_train = test_set_user[~test_set_user.index.isin(frame_test.index)]
print("TOTAL SAMPLES OF THE USER IN TRAIN SET: " + str(len(frame_train)))

TOTAL SAMPLES OF THE USER IN TRAIN SET: 7


In [144]:
assert len(frame_train) + len(frame_test) == len(test_set_user)

In [145]:
def add_testdata(traindf, test_set):
    """    
    :param traindf: dataframe que conté les dades de train
    :param test_set: dataframe que conté les dades de test

    :return : 
        - :param 1st: dataframe que conté les dades de train juntament amb el 80% de test seleccionat
        - :param 2nd: dataframe que conté les dades de test que queden (20% restant)
    """
    
    # Creem dos dataframes per guardar les dades noves
    new_train_set = traindf.copy()
    new_test_set = pd.DataFrame(columns=test_set.columns)
    
    # Obtenim el groupby_count com a l'exemple
    groupby_count = (test_set.groupby('user_id')['movie_id'].count()*0.2).reset_index()
   
    # Iterem per cada usuari
    for i in range(len(groupby_count)):
        # Obtenim la i-éssima fila
        data = groupby_count.iloc[i]

        # Guardem les pelis i l'usuari
        n_test_samples = int(data['movie_id'])
        u = data['user_id']

        # Guardem les dades d'aquest user
        test_set_user = test_set[test_set['user_id'] == u]

        # Creem els dos frames
        frame_test = test_set_user.sample(n_test_samples)
        frame_train = test_set_user[~test_set_user.index.isin(frame_test.index)]

        # Guardem el 80% en el new_train_set
        new_train_set = pd.concat([new_train_set, frame_train])

        # Guardem el 20% en el new_test_set
        new_test_set = pd.concat([new_test_set, frame_test])

    return new_train_set, new_test_set    

In [146]:
t = datetime.datetime.now()
train, test = add_testdata(train_set, test_set)
t = datetime.datetime.now()-t
print(str(t))

0:06:11.605498


In [147]:
train

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
0,0,1104,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,0,639,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,0,853,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,0,3177,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,0,2162,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy
...,...,...,...,...,...,...,...,...,...,...
999887,6039,855,5,957717557,M,25,6,11106,Roman Holiday (1953),Comedy|Romance
999900,6039,2855,3,956715805,M,25,6,11106,Adventures of Buckaroo Bonzai Across the 8th D...,Adventure|Comedy|Sci-Fi
1000107,6039,2422,5,957717158,M,25,6,11106,After Life (1998),Drama
1000008,6039,1782,3,956715569,M,25,6,11106,Driving Miss Daisy (1989),Drama


In [148]:
test

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
134,1,2512,3,978298196,M,56,16,70072,Ghostbusters II (1989),Comedy|Horror
202,2,2651,4,978297039,M,25,15,55117,American Beauty (1999),Comedy|Drama
410,4,858,4,978241072,M,25,20,55455,"Wizard of Oz, The (1939)",Adventure|Children's|Drama|Musical
394,4,2126,4,978242640,M,25,20,55455,Happiness (1998),Comedy
284,4,2495,4,978243085,M,25,20,55455,"South Park: Bigger, Longer and Uncut (1999)",Animation|Comedy
...,...,...,...,...,...,...,...,...,...,...
999917,6039,3189,3,997453909,M,25,6,11106,Animal House (1978),Comedy
1000023,6039,44,3,956704953,M,25,6,11106,To Die For (1995),Comedy|Drama
1000159,6039,3115,3,956715910,M,25,6,11106,Birdy (1984),Drama|War
999896,6039,869,4,957717274,M,25,6,11106,Notorious (1946),Film-Noir|Romance|Thriller


In [149]:
train.shape

(982648, 10)

In [150]:
test.shape

(17561, 10)

In [151]:
data.shape

(1000209, 10)

In [152]:
assert train.shape[0] + test.shape[0] == data.shape[0]

#### EXERCICI G.3

+ Fes una funció que serveixi per evaluar el nostre sistema usant la mètrica MAE. 

In [155]:
def evaluateRecommendations(train, test, m,n, sim):
    """
    Retorna l'error generat pel model
    
    :param DataFrame: dataframe que conté totes les dades
    :param userID: usuari respecte al qual fem la recomanació
    :param m: nombre d'usuaris que volem per fer la recomanació
    :param n: nombre de pelis a retornar (no)
    :param sim: matriu de similitud
    :return : Escalar (float) corresponent al MAE
    """  
    # Convertim les dades del train i el test al format correcte
    df_train = build_counts_table(train)
    df_test = build_counts_table(test)
    
    # Calculem la matriu de similituds
    sim = similarity_matrix_2(df_train)
    
    # La matriu d'errors guarda els errors de cada usuari
    error = []

    # Inicialitzem el comptador del total de valors
    N = 0

    # Iterem per tots els usuaris i files del test
    for user, row in df_test.iterrows():
        # Ens quedem amb les pel·lícules valorades per aquest usuari en el test
        row = pd.DataFrame(row).dropna()

        # Generem les prediccions per aquest usuari utilitzant dades del train
        predictions = weighted_average(df_train, user, sim, m)
        predictions = pd.DataFrame.from_dict(predictions, orient="index")

        # Trobar les pel·lícules comunes entre les valorades i les predites
        common_indices = row.index.intersection(predictions.index)

        # Filtrar les dades per només incloure les pel·lícules comunes
        row_filtered = row.loc[common_indices]
        predictions_filtered = predictions.loc[common_indices]
        
        # Fem el càlcul del MAE (veuere diapositiva 72 de teoria)
        result = abs(row_filtered.values - predictions_filtered.values)
        error.append(result.sum())
        N += len(result)

    # Calcular i retornar el MAE com la mitjana dels errors absoluts
    mae = sum(error) / N if N > 0 else float('inf')
    return mae

In [156]:
t = datetime.datetime.now()
mae = evaluateRecommendations(train, test, 50, 10, sim)
t = datetime.datetime.now()-t
print(str(t))

0:01:34.264879


In [157]:
mae

0.7633865196617512

In [ ]:
### Observació
### Hem intentat trobar usuaris i movies comunes per tal de poder 
### filtrar amb més seguretat els resultats, però hem tingut problemes amb 
### l'exercici H. Les línies eren les següents.

# common_users = df_train.index.intersection(df_test.index)
# df_test = df_test.loc[common_users]
# Filtrar usuaris comuns entre train i test
# common_users = df_train.index.intersection(df_test.index)
# df_train = df_train.loc[common_users]
# df_test = df_test.loc[common_users]
# # Filtrar pel·lícules comunes entre train i test
# common_movies = df_train.columns.intersection(df_test.columns)
# df_train = df_train[common_movies]
# df_test = df_test[common_movies]

### 3.8. EXERCICI H (exercici opcional, no obligatori)


+ **Que surt més a compte, fer un recomanador unic pels dos sexes o un per cada sexe?** Justifica la resposta per escrit i amb el codi necessari.

Un únic sistema de recomanació:

- Pros: Aprofita tota la base de dades per entrenar el model, fet que podria millorar la qualitat de les recomanacions considerant una major diversitat de dades.
- Contres: Podria no capturar diferències específiques en les preferències de gènere si aquestes són significatives.

Sistemes separats per sexe:

- Pros: Permeten capturar preferències específiques per a cada grup, fet que podria millorar les recomanacions si hi ha diferències significatives entre els gèneres.
- Contres: Menor quantitat de dades per entrenar cada model, cosa que podria afectar la precisió, especialment en grups petits.

1. Separar les dades en dos subconjunts:

       1.1. df_male: usuaris masculins.
       1.2. df_female: usuaris femenins.
   
3. Entrenar i avaluar:

       2.1. Model únic:
           2.1.1. Entrenar un únic sistema de recomanació amb totes les dades.
           2.1.2. Avaluar el MAE per a homes i dones per separat.
   
       2.2. Models separats:
           2.2.1. Entrenar un model per a homes (amb df_male).
           2.2.2. Entrenar un model per a dones (amb df_female).
           2.2.3. Avaluar el MAE de cada model en el seu respectiu grup.
   
4. Comparar els resultats de tots dos enfocaments:

       3.1. Determinar si el rendiment millora significativament separant els models.

In [161]:
# Separar les dades per sexe
df_male = train[train['gender'] == 'M']
df_female = train[train['gender'] == 'F']

In [162]:
# Avaluar el model únic
mae_all_male = evaluateRecommendations(train, test[test['gender'] == 'M'], 50, 10, sim)
mae_all_female = evaluateRecommendations(train, test[test['gender'] == 'F'], 50, 10, sim)

# Comparar resultats
print("MAE Model Únic - Homes:", mae_all_male)
print("MAE Model Únic - Dones:", mae_all_female)

MAE Model Únic - Homes: 0.7586890048682016
MAE Model Únic - Dones: 0.7781450038637184


In [ ]:
# Com déiem a l'observació anterior, aquest apartat ens ha donat problemes
# amb la funció evaluateRecommendations(). En particular, ara hauríem de fer
# un model per homes i un altre per dones. El codi seria semblant al següent.
# train_male = train[train['gender'] == 'M']
# train_female = train[train['gender'] == 'F']
# test_male = test[test['gender'] == 'M']
# test_female = test[test['gender'] == 'F']
# mae_only_male = evaluateRecommendations(train_male, test_male, 50, 10, sim)
# mae_only_female = evaluateRecommendations(train_female, test_female, 50, 10, sim)
# print(mae_only_male, mae_only_female)